# ใบงานที่ 3 (LAB 3): Model Comparison

โน้ตบุ๊กนี้รวบรวมและเปรียบเทียบโมเดลทั้งหมดจาก LAB 1 (Regression) และ LAB 2 (Classification) อย่างเป็นระบบ ครอบคลุม:

1. Simple vs Multiple Linear Regression
2. Training vs Testing Performance (ตรวจสอบ Overfitting/Underfitting)
3. Regression vs Classification (เปรียบเทียบเชิงแนวคิดและตัวชี้วัด)
4. Model Performance Metrics (ตารางสรุปผลรวมทุกโมเดล)

> **หมายเหตุ:** โน้ตบุ๊กนี้ฝึกโมเดลใหม่ทั้งหมดจาก `age_gender.csv` (pipeline เดียวกับ LAB 1–2) เพื่อให้สามารถเปรียบเทียบ
> Train vs Test ของทุกโมเดลได้ในที่เดียว วาง `age_gender.csv` ไว้โฟลเดอร์เดียวกับโน้ตบุ๊กก่อนรัน


In [ ]:
# ---- Import libraries ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

sns.set(style="whitegrid")
np.random.seed(42)


## 0. Data Preparation (ใช้ pipeline เดียวกับ LAB 1–2)

In [ ]:
DATA_PATH = "age_gender.csv"   # เปลี่ยน path ตรงนี้ถ้าไฟล์อยู่ที่อื่น
IMG_SIZE = 48

df = pd.read_csv(DATA_PATH)

def pixels_to_array(pixel_str):
    return np.array(pixel_str.split(), dtype='float32')

pixel_arrays = np.stack(df['pixels'].apply(pixels_to_array).values) / 255.0
age = df['age'].values
gender = df['gender'].values

print("Pixel matrix shape:", pixel_arrays.shape)
print("Rows:", len(df))


In [ ]:
def extract_stat_features(img_flat, size=IMG_SIZE):
    img = img_flat.reshape(size, size)
    h, w = size // 2, size // 2
    q1, q2, q3, q4 = img[:h,:w].mean(), img[:h,w:].mean(), img[h:,:w].mean(), img[h:,w:].mean()
    return [img.mean(), img.std(), img.min(), img.max(), q1, q2, q3, q4]

mean_intensity = pixel_arrays.mean(axis=1).reshape(-1, 1)
stat_features = np.array([extract_stat_features(row) for row in pixel_arrays])

print("Simple-feature matrix:", mean_intensity.shape)
print("Multiple-feature matrix:", stat_features.shape)


## 1. Simple vs Multiple Linear Regression

เปรียบเทียบโมเดล Regression สองแบบจาก LAB 1 บนชุด **test set เดียวกัน**:
- **Simple Linear Regression**: 1 feature (mean pixel intensity)
- **Multiple Linear Regression**: 8 features เชิงสถิติของภาพ

In [ ]:
# ---- แบ่ง train/test (ใช้ index เดียวกันสำหรับทั้งสองโมเดล เพื่อเทียบกันได้ตรง ๆ) ----
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=42
)

y_age_train, y_age_test = age[idx_train], age[idx_test]

# --- Simple LR ---
simple_model = LinearRegression().fit(mean_intensity[idx_train], y_age_train)
simple_train_pred = simple_model.predict(mean_intensity[idx_train])
simple_test_pred = simple_model.predict(mean_intensity[idx_test])

# --- Multiple LR ---
multi_model = LinearRegression().fit(stat_features[idx_train], y_age_train)
multi_train_pred = multi_model.predict(stat_features[idx_train])
multi_test_pred = multi_model.predict(stat_features[idx_test])

def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

simple_test_m = reg_metrics(y_age_test, simple_test_pred)
multi_test_m = reg_metrics(y_age_test, multi_test_pred)

compare_reg = pd.DataFrame({
    'Simple LR (1 feature)': simple_test_m,
    'Multiple LR (8 features)': multi_test_m,
}).T
compare_reg


In [ ]:
compare_reg[['RMSE', 'MAE']].plot(kind='bar', figsize=(6,4))
plt.title("Simple vs Multiple Linear Regression (Test Set)")
plt.ylabel("Years (error)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Multiple LR ลด RMSE ลง {simple_test_m['RMSE'] - multi_test_m['RMSE']:.2f} ปี "
      f"และเพิ่ม R^2 ขึ้น {multi_test_m['R2'] - simple_test_m['R2']:.3f} เมื่อเทียบกับ Simple LR")


## 2. Training vs Testing Performance

ตรวจสอบว่าโมเดลแต่ละตัว **overfit** (ดีบน train แต่แย่บน test) หรือ **underfit** (แย่ทั้งคู่) หรือไม่
โดยเปรียบเทียบ metric บน training set กับ test set ของทุกโมเดลใน LAB 1 และ LAB 2

In [ ]:
# ---- Regression models: Train vs Test ----
pca_reg = PCA(n_components=50, random_state=42)
scaler_reg = StandardScaler()

X_train_scaled_reg = scaler_reg.fit_transform(pixel_arrays[idx_train])
X_test_scaled_reg = scaler_reg.transform(pixel_arrays[idx_test])
X_train_pca_reg = pca_reg.fit_transform(X_train_scaled_reg)
X_test_pca_reg = pca_reg.transform(X_test_scaled_reg)

pca_reg_model = LinearRegression().fit(X_train_pca_reg, y_age_train)
pca_reg_train_pred = pca_reg_model.predict(X_train_pca_reg)
pca_reg_test_pred = pca_reg_model.predict(X_test_pca_reg)

reg_train_test = pd.DataFrame({
    'Simple LR - Train': reg_metrics(y_age_train, simple_train_pred),
    'Simple LR - Test': reg_metrics(y_age_test, simple_test_pred),
    'Multiple LR - Train': reg_metrics(y_age_train, multi_train_pred),
    'Multiple LR - Test': reg_metrics(y_age_test, multi_test_pred),
    'PCA+LR - Train': reg_metrics(y_age_train, pca_reg_train_pred),
    'PCA+LR - Test': reg_metrics(y_age_test, pca_reg_test_pred),
}).T
reg_train_test


In [ ]:
# ---- Classification models: Train vs Test ----
y_gender_train, y_gender_test = gender[idx_train], gender[idx_test]

scaler_clf = StandardScaler()
X_train_scaled_clf = scaler_clf.fit_transform(pixel_arrays[idx_train])
X_test_scaled_clf = scaler_clf.transform(pixel_arrays[idx_test])

baseline_clf = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled_clf, y_gender_train)
base_train_pred = baseline_clf.predict(X_train_scaled_clf)
base_test_pred = baseline_clf.predict(X_test_scaled_clf)
base_train_proba = baseline_clf.predict_proba(X_train_scaled_clf)[:, 1]
base_test_proba = baseline_clf.predict_proba(X_test_scaled_clf)[:, 1]

pca_clf_reducer = PCA(n_components=50, random_state=42)
X_train_pca_clf = pca_clf_reducer.fit_transform(X_train_scaled_clf)
X_test_pca_clf = pca_clf_reducer.transform(X_test_scaled_clf)

pca_clf = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_pca_clf, y_gender_train)
pca_train_pred = pca_clf.predict(X_train_pca_clf)
pca_test_pred = pca_clf.predict(X_test_pca_clf)
pca_train_proba = pca_clf.predict_proba(X_train_pca_clf)[:, 1]
pca_test_proba = pca_clf.predict_proba(X_test_pca_clf)[:, 1]

def clf_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_proba),
    }

clf_train_test = pd.DataFrame({
    'Baseline LogReg - Train': clf_metrics(y_gender_train, base_train_pred, base_train_proba),
    'Baseline LogReg - Test': clf_metrics(y_gender_test, base_test_pred, base_test_proba),
    'PCA+LogReg - Train': clf_metrics(y_gender_train, pca_train_pred, pca_train_proba),
    'PCA+LogReg - Test': clf_metrics(y_gender_test, pca_test_pred, pca_test_proba),
}).T
clf_train_test


In [ ]:
# ---- กราฟเปรียบเทียบ Train vs Test (Regression: R^2, Classification: Accuracy) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

reg_r2 = pd.DataFrame({
    'Train': [reg_train_test.loc['Simple LR - Train', 'R2'],
              reg_train_test.loc['Multiple LR - Train', 'R2'],
              reg_train_test.loc['PCA+LR - Train', 'R2']],
    'Test': [reg_train_test.loc['Simple LR - Test', 'R2'],
             reg_train_test.loc['Multiple LR - Test', 'R2'],
             reg_train_test.loc['PCA+LR - Test', 'R2']],
}, index=['Simple LR', 'Multiple LR', 'PCA+LR'])
reg_r2.plot(kind='bar', ax=axes[0])
axes[0].set_title("Regression: R² (Train vs Test)")
axes[0].set_ylabel("R^2")
axes[0].tick_params(axis='x', rotation=15)

clf_acc = pd.DataFrame({
    'Train': [clf_train_test.loc['Baseline LogReg - Train', 'Accuracy'],
              clf_train_test.loc['PCA+LogReg - Train', 'Accuracy']],
    'Test': [clf_train_test.loc['Baseline LogReg - Test', 'Accuracy'],
             clf_train_test.loc['PCA+LogReg - Test', 'Accuracy']],
}, index=['Baseline LogReg', 'PCA+LogReg'])
clf_acc.plot(kind='bar', ax=axes[1])
axes[1].set_title("Classification: Accuracy (Train vs Test)")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


**การอ่านผล:** ถ้า metric บน Train สูงกว่า Test มาก (ช่องว่างกว้าง) แสดงว่าโมเดล **overfit** กับข้อมูล training
ถ้าทั้ง Train และ Test ต่ำพร้อมกัน แสดงว่าโมเดล **underfit** (โมเดลง่ายเกินไปสำหรับความซับซ้อนของข้อมูล)
โดยทั่วไป Baseline Logistic Regression ที่ใช้ raw pixels (2,304 มิติ) มีความเสี่ยง overfit สูงกว่า PCA+LogReg
เนื่องจากจำนวน features ต่อจำนวนข้อมูลสูงกว่ามาก

## 3. Regression vs Classification

Regression และ Classification ต่างกันที่ **ประเภทของค่าที่ทำนาย** และ **ตัวชี้วัดที่ใช้ประเมินผล**

In [ ]:
concept_compare = pd.DataFrame({
    'ประเด็น': [
        'ประเภทของ Target',
        'ตัวอย่าง Target ในแล็บนี้',
        'ชนิดของค่าที่ทำนาย',
        'โมเดลที่ใช้',
        'ตัวชี้วัดหลัก',
        'ความหมายของ "ถูก/ผิด"',
    ],
    'Regression': [
        'ค่าต่อเนื่อง (Continuous)',
        'อายุ (Age) — ตัวเลขจำนวนเต็ม/ทศนิยม',
        'ตัวเลขในช่วงไม่จำกัด',
        'Linear Regression',
        'MSE, RMSE, MAE, R²',
        'ระยะห่างระหว่างค่าจริงกับค่าทำนาย (ยิ่งใกล้ยิ่งดี)',
    ],
    'Classification': [
        'หมวดหมู่ (Categorical)',
        'เพศ (Gender) — Male / Female',
        'คลาสจากจำนวนจำกัด',
        'Logistic Regression',
        'Accuracy, Precision, Recall, F1, ROC/AUC',
        'ทำนายตรงคลาสหรือไม่ (ถูก/ผิดชัดเจน)',
    ],
})
concept_compare


In [ ]:
# ---- เปรียบเทียบผลลัพธ์จริงของแล็บนี้ ----
print("LAB 1 (Regression - Age):   R^2 (test, PCA+LR)      =", round(reg_train_test.loc['PCA+LR - Test', 'R2'], 3))
print("LAB 2 (Classification - Gender): Accuracy (test, PCA+LogReg) =",
      round(clf_train_test.loc['PCA+LogReg - Test', 'Accuracy'], 3))
print()
print("แม้ใช้ pipeline การเตรียมข้อมูลและ PCA แบบเดียวกัน แต่ Classification (จำแนกเพศ) ทำได้แม่นยำกว่า")
print("Regression (ทำนายอายุ) อย่างชัดเจน เพราะ 'เพศ' มีลักษณะเด่นในภาพที่แยกสองกลุ่มได้ชัดกว่า")
print("ขณะที่ 'อายุ' เป็นสเปกตรัมต่อเนื่องที่ขึ้นกับปัจจัยซับซ้อนกว่าความสัมพันธ์เชิงเส้นจากพิกเซลภาพ")


## 4. Model Performance Metrics (สรุปรวมทุกโมเดล)

In [ ]:
# ---- ตารางสรุป Regression (Test set) ----
final_reg = reg_train_test.loc[['Simple LR - Test', 'Multiple LR - Test', 'PCA+LR - Test']]
final_reg.index = ['Simple Linear Regression', 'Multiple Linear Regression', 'PCA (50) + Linear Regression']
print("=== Regression Models (Age Prediction) — Test Set ===")
final_reg


In [ ]:
# ---- ตารางสรุป Classification (Test set) ----
final_clf = clf_train_test.loc[['Baseline LogReg - Test', 'PCA+LogReg - Test']]
final_clf.index = ['Logistic Regression (Raw Pixels)', 'PCA (50) + Logistic Regression']
print("=== Classification Models (Gender Prediction) — Test Set ===")
final_clf


In [ ]:
# ---- Export ตารางสรุปเป็น CSV สำหรับใส่ในรายงาน/README ----
final_reg.to_csv("lab1_regression_results.csv")
final_clf.to_csv("lab2_classification_results.csv")
print("Saved: lab1_regression_results.csv, lab2_classification_results.csv")


## 5. สรุปผลรวม (LAB 1–3)

- **Simple vs Multiple Linear Regression:** การเพิ่ม features เชิงสถิติ (จาก 1 → 8 features) ช่วยลด RMSE และเพิ่ม R²
  ได้เล็กน้อย แต่ยังห่างไกลจากการใช้ข้อมูลภาพทั้งหมดผ่าน PCA ซึ่งให้ผลลัพธ์ดีที่สุดในกลุ่ม Regression
- **Training vs Testing:** โมเดลทั้งหมดในแล็บนี้ไม่มีสัญญาณ overfitting ที่รุนแรง เนื่องจากใช้โมเดลเชิงเส้นพื้นฐาน
  (Linear/Logistic Regression) ที่มีความซับซ้อนต่ำ แต่ Baseline Logistic Regression บน raw pixels มีความเสี่ยง overfit
  มากกว่ารุ่นที่ใช้ PCA เพราะมีจำนวน features สูงกว่าจำนวนข้อมูลค่อนข้างมาก
- **Regression vs Classification:** ทั้งสองงานใช้ pipeline การเตรียมข้อมูลและ PCA แบบเดียวกัน แต่ **Classification
  (จำแนกเพศ) ให้ผลลัพธ์แม่นยำกว่า Regression (ทำนายอายุ)** อย่างชัดเจน เพราะลักษณะภาพที่บ่งบอกเพศมีความชัดเจนกว่า
  ความสัมพันธ์เชิงเส้นระหว่างภาพกับอายุ
- **Model Performance Metrics:** ตัวชี้วัดของ Regression (MSE/RMSE/MAE/R²) และ Classification (Accuracy/Precision/
  Recall/F1/AUC) ไม่สามารถเทียบกันตรง ๆ ได้ เนื่องจากวัดคุณสมบัติของโมเดลคนละประเภท แต่สามารถใช้เพื่อเปรียบเทียบ
  **ภายในกลุ่มงานเดียวกัน** เพื่อเลือกโมเดลที่เหมาะสมที่สุดสำหรับแต่ละปัญหา

### สรุปโมเดลที่แนะนำ
| งาน | โมเดลที่แนะนำ | เหตุผล |
|---|---|---|
| Age Prediction (Regression) | PCA (50) + Linear Regression | RMSE ต่ำสุด, R² สูงสุดในกลุ่ม Regression |
| Gender Prediction (Classification) | PCA (50) + Logistic Regression | Accuracy/AUC ใกล้เคียง baseline แต่ลดความเสี่ยง overfitting และเวลาฝึกโมเดล |
